In [1]:
# отбираем данные для дообучения yandexgpt размером 50 тыс. строк для сравнения с классификацией
# отбираем две тестовые выборки - на 5000 строк с реальным распределением классов и на 900 строк с равным количеством строк каждого класса (по 100)
# эти же наборы использовались для проверки классификации на chatgpt и gemini AI studio


In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option("display.max_colwidth", None)

data_download = pd.read_csv("../../uc_data_labeling/data_download_uc.csv")

In [3]:
# поскольку дообучение классификации тексов возможно в формате "строка:метка", 
# отбираем строки где нет пропусков в назначениях (будем дообучать по ним) и в целевом признаке 
data_download.dropna(subset=['universal_category', 'purpose'], inplace=True)

In [4]:
data_download.head(2)


,id,account_id,contractor_id,date,payments_amount,purpose,article_id,expenditure,project_id,counterpartie_id,donor_id,robot_id,donor_cat_id,accounts__id,accounts__user_id,articles__id,articles__user_id,articles__parent_id,projects__id,projects__user_id,projects__parent_id,counterparties__id,counterparties__user_id,counterparties__parent_id,robots__id,robots__user_id,article_alternative_names__user_id,article_name,universal_category
125,4913,19,75,2022-09-12,700.00,Перевод денежных средств по договору присоединения к условиям оказания услуг ИТО при осуществлении переводов денежных средств благотворительным организациям от 26/05/2020. Без учета НДС.,822,incoming,0,278,150,0,0,NaN,NaN,822.0,45.0,821.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,пожертвование,пожертвования от физических лиц (напрямую)
151,4979,19,76,2022-09-19,1088.64,//Реестр// Количество 9. Перечисление денежных средств по договору НЭК.193578.01 по реестру за 18.09.2022. Без НДС,822,incoming,0,278,122,0,0,NaN,NaN,822.0,45.0,821.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,пожертвование,пожертвования от физических лиц (напрямую)


In [5]:
# разбиваем частично под дообучение yandexgpt

X = data_download['purpose']
y = data_download['universal_category']

X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=50000,test_size=100, random_state=42, stratify=y)


In [6]:
rest = data_download[~(data_download.index.isin(X_train.index) | data_download.index.isin(X_test.index))]

In [7]:
# из остатка rest отбираем по 100 строк для второго тестового набора
balanced_sample = (
    rest
    .groupby("universal_category", group_keys=False)
    .apply(lambda x: x.sample(n=100, random_state=42, replace=False))
)

balanced_sample = balanced_sample.sample(frac=1, random_state=42)

X_test_balanced = balanced_sample['purpose']
y_test_balanced = balanced_sample['universal_category']

/var/folders/yf/tnhxwgts2035718d0nhj30rw0000gn/T/ipykernel_23522/2078571289.py:5: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=100, random_state=42, replace=False))


In [8]:
rest_1 = rest[~rest.index.isin(balanced_sample.index)]

In [9]:
# проверим размерности
display(data_download.id.count(),
        X_train.count(),
        X_test.count(),
        balanced_sample.id.count(),
        rest_1.id.count())

541504

50000

100

900

490504

In [10]:
display(data_download.universal_category.value_counts(),
        y_train.value_counts(),
        y_test.value_counts(),
        rest_1.universal_category.value_counts())

universal_category
пожертвования от физических лиц (напрямую)     323709
пожертвования через платформы                  179298
пожертвования от юридических лиц (напрямую)     11180
прочие недоходные операции                       9447
продажа услуг                                    8450
продажа товаров                                  5953
финансовые доходы                                2172
членские и учредительские взносы                  850
гранты субсидии конкурсы                          445
Name: count, dtype: int64

universal_category
пожертвования от физических лиц (напрямую)     29890
пожертвования через платформы                  16556
пожертвования от юридических лиц (напрямую)     1032
прочие недоходные операции                       872
продажа услуг                                    780
продажа товаров                                  550
финансовые доходы                                201
членские и учредительские взносы                  78
гранты субсидии конкурсы                          41
Name: count, dtype: int64

universal_category
пожертвования от физических лиц (напрямую)     60
пожертвования через платформы                  33
прочие недоходные операции                      2
продажа услуг                                   2
пожертвования от юридических лиц (напрямую)     2
продажа товаров                                 1
Name: count, dtype: int64

universal_category
пожертвования от физических лиц (напрямую)     293659
пожертвования через платформы                  162609
пожертвования от юридических лиц (напрямую)     10046
прочие недоходные операции                       8473
продажа услуг                                    7568
продажа товаров                                  5302
финансовые доходы                                1871
членские и учредительские взносы                  672
гранты субсидии конкурсы                          304
Name: count, dtype: int64

In [11]:
# редкие классы в трейне и тесте представлены очень незначительно, поэтому все значения малых классов, которые не попали в трейн или оба теста добавим в трейн, и немного сократим мажоритарные класс - там даже несколько тысяч значений не сыграют роли

issues = {'гранты субсидии конкурсы': 304,
          'членские и учредительские взносы': 672,
          'финансовые доходы': 1871,
          'продажа товаров':3590,
          'продажа услуг':4544,
          'прочие недоходные операции':3526,
          'пожертвования от юридических лиц (напрямую)':7593}

for key,value in issues.items():
    try:
        temp_x = rest_1[rest_1['universal_category'] == key]['purpose'].sample(value, random_state=42)
        temp_y = rest_1[rest_1.index.isin(temp_x.index)]['universal_category']
    except ValueError as e:
        print(f"Ошибка на key='{key}': {e}")
        break
    X_train = pd.concat([X_train, temp_x],axis=0)
    y_train = pd.concat([y_train, temp_y], axis=0)


#drop_idx = y_train[y_train == 'пожертвования от физических лиц (напрямую)'].sample(sum(issues.values()), random_state=42).index

drop_idx = y_train[y_train.isin([
    'пожертвования от физических лиц (напрямую)',
    'пожертвования через платформы'
])].sample(sum(issues.values()), random_state=42).index

y_train = y_train.drop(drop_idx)
X_train = X_train.drop(drop_idx)

display(y_train.count(),
        y_train.value_counts())


50000

universal_category
пожертвования от физических лиц (напрямую)     15596
пожертвования через платформы                   8750
пожертвования от юридических лиц (напрямую)     8625
продажа услуг                                   5324
прочие недоходные операции                      4398
продажа товаров                                 4140
финансовые доходы                               2072
членские и учредительские взносы                 750
гранты субсидии конкурсы                         345
Name: count, dtype: int64

In [12]:
# сохраняем это все добро

X_train.to_csv('X_train.csv')
y_train.to_csv('y_train.csv')
X_test.to_csv('X_test.csv')
y_test.to_csv('y_test.csv')
X_test_balanced.to_csv('X_test_balanced.csv')
y_test_balanced.to_csv('y_test_balanced.csv')


In [13]:
# для обучения yandexgpt сохраним в json
train_json = pd.DataFrame({
    "text": X_train,
    "label": y_train
})

train_json = pd.concat([train_json["text"], pd.get_dummies(train_json["label"]).astype(int)], axis=1)
train_json.to_json("train.json", orient="records", lines=True, force_ascii=False)
